In [13]:
from pathlib import Path
import pandas as pd

In [14]:
catalog_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Earthquake_Catalog/")  # adjust to your path
for f in sorted(catalog_dir.glob("*.csv")):
    df = pd.read_csv(f)
    print(f"{f.stem:<30} {len(df):>8} events")

Central_Chile                      2597 events
Central_Nepal                       409 events
Central_Turkey                      750 events
Kanto_Japan                        3149 events
Kuch_India                           32 events
North_Island_NZ                    1470 events
Ordos_China                          30 events
Sichuan_China                      1154 events
Southern_Norway                       4 events
Southern_Sumatra_Indonesia         1642 events
Tohoku_Japan                       3898 events
Western_Australia                    54 events


In [15]:
for f in sorted(catalog_dir.glob("*.csv")):
    df = pd.read_csv(f)
    if len(df) < 100:
        print(f"\n{'='*40}")
        print(f"{f.stem}")
        print(f"  Total events: {len(df)}")
        if len(df) > 0:
            print(f"  Mag range:    {df['mag'].min():.1f} – {df['mag'].max():.1f}")
            print(f"  Date range:   {df['time'].min()[:10]} – {df['time'].max()[:10]}")
            print(f"  Mag distribution:")
            print(df['mag'].round(0).value_counts().sort_index().to_string())


Kuch_India
  Total events: 32
  Mag range:    3.7 – 5.5
  Date range:   2006-02-03 – 2025-12-25
  Mag distribution:
mag
4.0    23
5.0     7
6.0     2

Ordos_China
  Total events: 30
  Mag range:    3.7 – 4.5
  Date range:   2005-11-16 – 2024-04-25
  Mag distribution:
mag
4.0    30

Southern_Norway
  Total events: 4
  Mag range:    2.5 – 3.6
  Date range:   2005-07-28 – 2012-03-14
  Mag distribution:
mag
2.0    2
3.0    1
4.0    1

Western_Australia
  Total events: 54
  Mag range:    2.5 – 4.6
  Date range:   2005-05-12 – 2025-07-27
  Mag distribution:
mag
2.0     7
3.0    24
4.0    21
5.0     2


In [16]:
f = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Earthquake_Catalog/"+"Central_Turkey.csv")
df = pd.read_csv(f)

print(f"Total events: {len(df)}")
print(f"Mag range: {df['mag'].min():.1f} – {df['mag'].max():.1f}")
print(f"Date range: {df['time'].min()[:10]} – {df['time'].max()[:10]}")
print(f"\nMag distribution:")
print(df['mag'].round(0).value_counts().sort_index())
print(f"\nEvents per year:")
df['year'] = pd.to_datetime(df['time']).dt.year
print(df['year'].value_counts().sort_index())
print(df[df['mag'] >= 7.0][['time', 'mag', 'place']].to_string())

Total events: 750
Mag range: 2.7 – 7.8
Date range: 2005-07-24 – 2026-04-27

Mag distribution:
mag
3.0     38
4.0    556
5.0    142
6.0     11
7.0      1
8.0      2
Name: count, dtype: int64

Events per year:
year
2005     10
2006     27
2007     17
2008     10
2009      3
2010      3
2011      3
2012     14
2013      8
2014      5
2015      7
2016      3
2017      4
2018      7
2019      8
2020     27
2021      7
2022      9
2023    520
2024     42
2025     10
2026      6
Name: count, dtype: int64
                         time  mag                                                   place
486  2023-02-06T10:24:48.811Z  7.5  Elbistan earthquake, Kahramanmaras earthquake sequence
575  2023-02-06T01:17:34.342Z  7.8  Pazarcik earthquake, Kahramanmaras earthquake sequence


## Insights

<p> 
Catalog retrieval from USGS ComCat across all 12 patches reveals a clear divide between tectonically active and stable/low-seismicity regions that has direct implications for model design and evaluation. The seven high-seismicity patches: Kanto, Tohoku, Chile, Turkey, New Zealand, Sumatra, and Sichuan, returned between 750 and 3,898 events over the 2000–2024 study period, providing sufficient catalog density for temporal feature engineering, b-value estimation, and GNN training. Nepal sits at the lower end of this group with 409 events, reflecting sparse instrumentation rather than low seismicity, and is borderline for fine-tuning but usable as a transfer target.
</p>

<p>
The four sparse patches tell a more nuanced story. Western Australia (54 events) and Kutch India (32 events) are not failures of the retrieval — they reflect genuine physical conditions. Western Australia is a stable Archean craton with near-zero tectonic seismicity, and the 54 events retrieved represent a real, if thin, catalog usable for zero-shot and minimal few-shot evaluation. Kutch is more interesting: the sparse post-2005 catalog reflects post-mainshock quiescence following the 2001 Bhuj Mw 7.7 event, which falls outside the study window. The region is geologically primed for future hazard but seismically quiet in the observation period — a physically meaningful signal that the frozen geological prior should capture even when the temporal catalog cannot. Ordos China (30 events, all Mw ~4.0) has essentially no magnitude variation, making temporal feature engineering impossible, but its static geological signature as a stable North China craton interior is well-defined and valuable for clustering. Southern Norway (4 events) is too sparse for any modeling role and will be retained only in the geological prior construction and dataset paper.
</p>

<p>
The Turkey catalog deserves specific mention. With 750 total events, 520 of them fall in 2023 alone. It is a direct signature of the Kahramanmaraş Mw 7.8 and 7.5 sequence in February of that year. This dramatic rate change, from single-digit annual counts in quiet years to 520 in 2023, is precisely the kind of temporal signal the GNN backbone is designed to encode. It also makes Turkey the natural retrospective case study: a model trained on pre-2023 data and transferred from other patches should, if the frozen geological prior is working, assign elevated hazard to the East Anatolian Fault zone without ever having seen a Turkish earthquake sequence of this magnitude.
</p>

<p>
Across all patches, the catalog inspection confirms the core design assumption of the project: seismic catalog density is not a reliable proxy for geological hazard, and a model that relies solely on temporal catalog features will systematically underestimate risk in data-sparse but geologically active regions. The frozen prior exists precisely to carry information in the gap between what the catalog says and what the geology implies.
</p>